Creamos el cliente de openai

In [4]:
from openai import OpenAI

openai_client = OpenAI()

Definimos la estructura con pydantic
Estamos definiendo un evento de calendario. Extiende de BaseModel.
BaseModel viene con pydantic que es una librería de validaciíón de datos. Valida que los objetos de json se ajustan a unos criterios

In [1]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

Podemos ver como queda su esquema json

In [2]:
CalendarEvent.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object'}

El sdk de open ahi tiene soporte para salidas estructuradas.
En vez de usar responses.create usamos responses.parse y pasamos el formato de texto con nuestro objeto de pydantic

In [5]:
messages = [
    {"role": "system", "content": "Extract the event information."},
    {
        "role": "user",
        "content": "Alice and Bob are going to a science fair on Friday.",
    },
]

response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=messages,
    text_format=CalendarEvent,
)

Ahora vamos a acceder a los datos parseados.
Primero vamos por el path directo

In [6]:
response.output[0].content[0].parsed

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

Podemos ver la representación en raw

In [7]:
response.output[0].content[0].text

'{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'

También hay una propiedad atajo

In [8]:
event = response.output_parsed
event

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

Ahora lo comparamos con la salida desestructurada donde:
El primer intento puede venir asi:
Event: Science Fair
Participants: Alice and Bob
Day: Friday

Y el sgundo asi:
Event Information:
- Event: Science Fair
- Participants: Alice and Bob
- Day: Friday


In [9]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
)
print(response.output_text)

Event: Science Fair  
Participants: Alice and Bob  
Day: Friday  
